# 05 — SSGF Geometry Learning (Concept Demo)

Demonstrates the Stochastic Synthesis of Geometric Fields (SSGF)
outer cycle that learns coupling geometry from Kuramoto phase dynamics.

> **Note:** SSGF is part of the SCPN framework. This notebook shows the
> mathematical concept using a minimal self-contained implementation.
> For the full SSGF engine, see the SCPN-CODEBASE repository.

In [ ]:
import numpy as np
from dataclasses import dataclass, field


@dataclass
class SSGFConfig:
    N: int = 16
    z_dim: int = 10
    lr_z: float = 0.02
    outer_steps: int = 30
    micro_steps: int = 20
    sigma_g: float = 0.3
    dt: float = 0.05


class SSGFEngine:
    """Minimal SSGF: learn coupling geometry W from Kuramoto microcycles."""

    def __init__(self, cfg: SSGFConfig):
        self.cfg = cfg
        rng = np.random.default_rng(42)
        self.z = rng.normal(0, 0.1, cfg.z_dim)
        self.omega = rng.uniform(0.8, 1.5, cfg.N)
        self.theta = rng.uniform(0, 2 * np.pi, cfg.N)
        n_upper = cfg.N * (cfg.N - 1) // 2
        self._z_to_w_idx = np.triu_indices(cfg.N, k=1)
        if cfg.z_dim < n_upper:
            self._proj = rng.normal(0, 1 / np.sqrt(cfg.z_dim), (cfg.z_dim, n_upper))

    def _decode_W(self):
        n_upper = self.cfg.N * (self.cfg.N - 1) // 2
        if self.cfg.z_dim >= n_upper:
            raw = self.z[:n_upper]
        else:
            raw = self.z @ self._proj
        W = np.zeros((self.cfg.N, self.cfg.N))
        W[self._z_to_w_idx] = np.log1p(np.exp(raw))  # softplus
        W = W + W.T
        return W

    def _micro_step(self, W):
        N, dt = self.cfg.N, self.cfg.dt
        for _ in range(self.cfg.micro_steps):
            diff = self.theta[np.newaxis, :] - self.theta[:, np.newaxis]
            coupling = (W * np.sin(diff)).sum(axis=1) / max(N - 1, 1)
            self.theta += dt * (self.omega + self.cfg.sigma_g * coupling)

    def _order_param(self):
        return float(np.abs(np.mean(np.exp(1j * self.theta))))

    def _cost(self, W):
        R = self._order_param()
        c_micro = 1.0 - R
        reg = 0.01 * np.sum(W**2)
        return c_micro + reg, c_micro, R

    def run(self):
        results = []
        for step in range(self.cfg.outer_steps):
            W = self._decode_W()
            self._micro_step(W)
            u_total, c_micro, R = self._cost(W)
            results.append({"u_total": u_total, "c_micro": c_micro, "R_global": R})
            # Finite-difference gradient on z
            grad = np.zeros_like(self.z)
            eps = 1e-4
            for i in range(len(self.z)):
                self.z[i] += eps
                W_p = self._decode_W()
                c_p = 0.01 * np.sum(W_p**2) + (1.0 - self._order_param())
                self.z[i] -= 2 * eps
                W_m = self._decode_W()
                c_m = 0.01 * np.sum(W_m**2) + (1.0 - self._order_param())
                self.z[i] += eps
                grad[i] = (c_p - c_m) / (2 * eps)
            self.z -= self.cfg.lr_z * grad
        return results


print("SSGFEngine ready")

## Initialise Engine

In [ ]:
cfg = SSGFConfig(N=16, z_dim=10, lr_z=0.02, outer_steps=30, micro_steps=20)
engine = SSGFEngine(cfg)
print(f"Engine initialised: N={cfg.N}, z_dim={cfg.z_dim}")

## Run Outer Cycle
Watch costs decrease as geometry stabilises the microcycles.

In [ ]:
results = engine.run()

print(f"\nConvergence over {len(results)} outer steps:")
print(f"  U_total: {results[0]['u_total']:.4f} → {results[-1]['u_total']:.4f}")
print(f"  C_micro: {results[0]['c_micro']:.4f} → {results[-1]['c_micro']:.4f}")
print(f"  R_global: {results[0]['R_global']:.4f} → {results[-1]['R_global']:.4f}")

## Visualise Cost Convergence

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    steps = range(len(results))
    ax1.plot(steps, [r["u_total"] for r in results], "b-", label="U_total")
    ax1.plot(steps, [r["c_micro"] for r in results], "r--", label="C_micro")
    ax1.set_xlabel("Outer step")
    ax1.set_ylabel("Cost")
    ax1.legend()
    ax1.set_title("Cost Convergence")
    ax2.plot(steps, [r["R_global"] for r in results], "g-")
    ax2.set_xlabel("Outer step")
    ax2.set_ylabel("R_global")
    ax2.set_title("Order Parameter")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for plots: pip install matplotlib")